In [1]:
import sys
import os
import logging
import gc
import time
import torch
import warnings
import psutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gliner import GLiNER
from gliner.data_processing.collator import DataCollator
from gliner.training import Trainer, TrainingArguments
from transformers import TrainerCallback
from peft import LoraConfig, get_peft_model, TaskType,PeftModel

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)
sys.path

/opt/app/notebooks/abhishek/active_gliner/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python311.zip',
 '/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python3.11',
 '/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python3.11/lib-dynload',
 '',
 '/opt/app/notebooks/abhishek/active_gliner/.venv/lib/python3.11/site-packages',
 '/tmp/tmp0huezdup']

In [2]:
src_path=os.path.join(os.path.dirname(os.getcwd()),'src')
sys.path.append(src_path)
sys.path

['/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python311.zip',
 '/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python3.11',
 '/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python3.11/lib-dynload',
 '',
 '/opt/app/notebooks/abhishek/active_gliner/.venv/lib/python3.11/site-packages',
 '/tmp/tmp0huezdup',
 '/opt/app/notebooks/abhishek/active_gliner/src']

In [3]:
from config.settings import Settings
settings = Settings()

print(f"Settings cache_dir: {settings.cache_dir}")
print(f"Cache absolute path: {settings.cache_dir.resolve()}")
print(f"Does cache dir contain 'notebooks': {'notebooks' in str(settings.cache_dir)}")

print("=== Integration Test ===")
from utils.logging import setup_logging
from utils.reproducibility import set_all_seeds
from utils.device import setup_device

# Complete setup like your original code
settings = Settings()
settings.setup()  # Apply environment and create directories

logger = setup_logging(log_dir=str(settings.logs_dir))
set_all_seeds(seed=settings.global_seed, logger=logger)
device = setup_device(logger=logger)

logger.info("All modules integrated successfully!")
print(f"Final setup: seed={settings.global_seed}, device={device}, batch_size={settings.batch_size}")



INFO:ActiveLearning:================================================================================
INFO:ActiveLearning:ACTIVE LEARNING PIPELINE WITH PROPER TRAIN/TEST SEPARATION
INFO:ActiveLearning:================================================================================
INFO:ActiveLearning:Log file: /opt/app/notebooks/abhishek/active_gliner/logs/active_learning_20250910_210404.log
INFO:ActiveLearning:Setting all seeds to 42 for reproducibility...
INFO:ActiveLearning:Using device: cuda
INFO:ActiveLearning:CUDA version: 12.8


INFO:ActiveLearning:Number of GPUs visible: 1
INFO:ActiveLearning:Current GPU: 0
INFO:ActiveLearning:GPU Name: NVIDIA GeForce RTX 3090
INFO:ActiveLearning:GPU Memory: 23.6 GB
INFO:ActiveLearning:All modules integrated successfully!


Settings cache_dir: /opt/app/notebooks/abhishek/active_gliner/cache
Cache absolute path: /opt/app/notebooks/abhishek/active_gliner/cache
Does cache dir contain 'notebooks': True
=== Integration Test ===
Final setup: seed=42, device=cuda, batch_size=8


In [4]:
from data.loader import load_json_file,load_mit_dataset
from data.transforms import get_ner_statistics
from selection.strategies import get_lowest_score_examples_sorted

low_score_60_examples=load_json_file("../results/low_score_1000_examples.json")
syn_57_examples=load_json_file("../results/syn_1000_examples.json")
test_data,entity_types=load_mit_dataset("../data/mit-movie/test.json","../data/mit-movie/labels.json")

low_score_stats=get_ner_statistics(low_score_60_examples,entity_types)
print(low_score_stats)


syn_stats=get_ner_statistics(syn_57_examples,entity_types)
print(syn_stats)


Loading train data from: ../data/mit-movie/test.json
Processed 2442 examples
Entity types: ['genre', 'year', 'plot', 'average ratings', 'actor', 'title', 'song', 'character', 'rating', 'review', 'director', 'trailer']
{'total_examples': 1000, 'avg_num_tokens': 11.3, 'avg_num_entities': 2.568, 'total_entities': 2568, 'unique_entity_types': 12, 'entity_type_counts': Counter({'genre': 565, 'year': 430, 'actor': 354, 'average ratings': 287, 'plot': 253, 'director': 224, 'rating': 224, 'title': 165, 'character': 20, 'review': 19, 'song': 14, 'trailer': 13}), 'entity_type_coverage': {'genre': 565, 'year': 430, 'plot': 253, 'average ratings': 287, 'actor': 354, 'title': 165, 'song': 14, 'character': 20, 'rating': 224, 'review': 19, 'director': 224, 'trailer': 13}}
{'total_examples': 898, 'avg_num_tokens': 69.08685968819599, 'avg_num_entities': 7.987750556792873, 'total_entities': 7173, 'unique_entity_types': 12, 'entity_type_counts': Counter({'title': 1065, 'actor': 1014, 'genre': 941, 'year'

In [5]:
# syn_train=syn_57_examples[0:50]
# syn_val=syn_57_examples[50:]
# len(syn_train)




In [6]:

print("=== Testing Evaluation Evaluator ===")

def intialize_model():



    model = GLiNER.from_pretrained("knowledgator/modern-gliner-bi-large-v1.0")
    model.config.max_len = 8192

    if hasattr(model.data_processor, 'transformer_tokenizer'):    
        model.data_processor.transformer_tokenizer.model_max_length = 8192

    # Get base parameter count
    base_total = sum(p.numel() for p in model.model.parameters())
    logger.info(f"Base Parameters: {base_total:,}")

    print("\n🔧 Applying FIXED LoRA Configuration...")

    # FIXED LoRA config - back to user's preferred values
    lora_config = LoraConfig(
        r=32,               # Back to 32 as requested
        lora_alpha=64,      # Back to 64 as requested
        target_modules=[
            # "query_proj", 
            # "key_proj",
            # "value_proj",
            "dense",
            "projection",
            "Wqkv", "Wo", "Wi",
            #   "linear_1", "linear_2",
            "query", "key", "value",  # BERT attention
        "intermediate.dense", "output.dense",  # BERT MLP,
        
        "span_rep_layer.span_rep_layer.project_start.3","span_rep_layer.span_rep_layer.project_start.0",
        "span_rep_layer.span_rep_layer.project_end.3","span_rep_layer.span_rep_layer.project_end.0",
        "span_rep_layer.span_rep_layer.out_project.3","span_rep_layer.span_rep_layer.out_project.0",
        'prompt_rep_layer.3','prompt_rep_layer.0',
        

        ],
        modules_to_save=[
                # "span_rep_layer",
            # "prompt_rep_layer"   # Only this one works properly
        ],
        lora_dropout=0.1,   # Reduced from 0.2
        bias="none",
        task_type=TaskType.TOKEN_CLS
    )

    # Apply LoRA
    model.model = get_peft_model(model.model, lora_config)

    # Manually make span_rep_layer trainable
    # for param in model.model.base_model.span_rep_layer.parameters():
    #     param.requires_grad = True

    print("✅ LoRA applied successfully!")

    # Get LoRA parameter count
    lora_trainable = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
    print(f"📊 Trainable Parameters: {lora_trainable:,} ({100*lora_trainable/base_total:.1f}% of original)")

    model.to(device)

    print("Model after lora")
    
    return model

model=intialize_model()
display(model)

=== Testing Evaluation Evaluator ===


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 5990.91it/s]
INFO:ActiveLearning:Base Parameters: 529,845,248



🔧 Applying FIXED LoRA Configuration...
✅ LoRA applied successfully!
📊 Trainable Parameters: 20,815,872 (3.9% of original)
Model after lora


GLiNER(
  (model): PeftModelForTokenClassification(
    (base_model): LoraModel(
      (model): SpanModel(
        (token_rep_layer): BiEncoder(
          (bert_layer): Transformer(
            (model): ModernBertModel(
              (embeddings): ModernBertEmbeddings(
                (tok_embeddings): Embedding(50368, 1024, padding_idx=50283)
                (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
                (drop): Dropout(p=0.0, inplace=False)
              )
              (layers): ModuleList(
                (0): ModernBertEncoderLayer(
                  (attn_norm): Identity()
                  (attn): ModernBertAttention(
                    (Wqkv): lora.Linear(
                      (base_layer): Linear(in_features=1024, out_features=3072, bias=False)
                      (lora_dropout): ModuleDict(
                        (default): Dropout(p=0.1, inplace=False)
                      )
                      (lora_A): ModuleDict(
                   

In [7]:
# ===============================================================================
# 3. Simple Training Monitor with Resource Tracking
# ===============================================================================

class SimpleTrainingMonitor(TrainerCallback):
    """Simple training monitor with resource tracking"""
    
    def __init__(self, patience=10):
        self.train_losses = []
        self.eval_losses = []
        self.learning_rates = []
        self.steps = []
        self.eval_steps = []
        self.patience = patience
        self.best_loss = float('inf')
        self.patience_counter = 0
        
        # Resource trackingPeftModel
        self.gpu_memory = []
        self.cpu_memory = []
        self.timestamps = []
        self.start_time = time.time()
        
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            if 'loss' in logs:
                self.train_losses.append(logs['loss'])
                self.steps.append(state.global_step)
                
                # Track resources
                current_time = (time.time() - self.start_time) / 60  # minutes
                self.timestamps.append(current_time)
                
                if torch.cuda.is_available():
                    gpu_mem = torch.cuda.memory_allocated() / 1024**3  # GB
                    self.gpu_memory.append(gpu_mem)
                
                cpu_mem = psutil.virtual_memory().percent
                self.cpu_memory.append(cpu_mem)
                
            if 'learning_rate' in logs:
                self.learning_rates.append(logs['learning_rate'])

    def on_step_begin(self, args, state, control, **kwargs):
      if state.global_step % 50 == 0:  # Every 50 steps
          torch.cuda.empty_cache()
          gc.collect()
    
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is not None and 'eval_loss' in metrics:
            eval_loss = metrics['eval_loss']
            
            # Check for NaN - CRITICAL FIX
            if np.isnan(eval_loss) or np.isinf(eval_loss):
                print(f"🚨 NaN validation loss detected! Stopping training.")
                control.should_training_stop = True
                return
                
            self.eval_losses.append(eval_loss)
            self.eval_steps.append(state.global_step)
            
            if eval_loss < self.best_loss:
                self.best_loss = eval_loss
                self.patience_counter = 0
                print(f"🎯 New best validation loss: {eval_loss:.4f}")
            else:
                self.patience_counter += 1
                print(f"📈 Validation loss: {eval_loss:.4f} | Patience: {self.patience_counter}/{self.patience}")
                
            if self.patience_counter >= self.patience:
                print("🛑 Early stopping triggered!")
                control.should_training_stop = True

# ===============================================================================
# 4. FIXED Training Configuration
# ===============================================================================

print("\n⚙️ FIXED Training Configuration...")

# FIXED training config - conservative but with user's preferred LR
training_config = {
    'num_steps': 1000,           # Reduced for stability
    'train_batch_size': 8,       # Increased batch size
    'gradient_accumulation_steps': 1,  # Reduced accumulation
    'learning_rate': 0.00021008343694753508,       # Starting with 1 as requested - conservative
    'others_lr': 0.00021008343694753508,           # Even lower for LoRA params
    'warmup_ratio': 0.07064690788186724,        # Longer warmup
    'eval_steps': 100,            # More frequent evaluation
    'save_steps': 100,
    'logging_steps': 10,         # More frequent logging
    'patience': 5,              # More patience
    'max_grad_norm': 1,        # CRITICAL: Gradient clipping
}

print(f"📋 Effective batch size: {training_config['train_batch_size'] * training_config['gradient_accumulation_steps']}")
print(f"📋 Total training steps: {training_config['num_steps']}")
print(f"🔥 CRITICAL FIXES APPLIED:")
print(f"   • Learning rate: {training_config['learning_rate']} (conservative)")
print(f"   • LoRA r: 32, alpha: 64 (as requested)")
# print(f"   • Gradient clipping: {training_config['max_grad_norm']}")
print(f"   • FP16: DISABLED for stability")

# Setup data collator
data_collator = DataCollator(
    model.config, 
    data_processor=model.data_processor, 
    prepare_labels=True
)

# Initialize training monitor
monitor = SimpleTrainingMonitor(patience=training_config['patience'])

# FIXED Training arguments
training_args = TrainingArguments(
    output_dir="../models/syn_model",
    learning_rate=training_config['learning_rate'],    # FIXED: Much lower
    weight_decay=0.020216630535603918,                                # Reduced weight decay
    others_lr=training_config['others_lr'],            # FIXED: Much lower
    others_weight_decay=0.020216630535603918,
    lr_scheduler_type="cosine",                        # Changed from linear
    warmup_ratio=training_config['warmup_ratio'],
    per_device_train_batch_size=training_config['train_batch_size'],
    per_device_eval_batch_size=training_config['train_batch_size'],
    gradient_accumulation_steps=training_config['gradient_accumulation_steps'],
    max_steps=training_config['num_steps'],
    max_grad_norm=training_config['max_grad_norm'],    # CRITICAL: Added gradient clipping
    
    # FIXED focal loss - much more conservative
    focal_loss_alpha=0.75,      
    focal_loss_gamma=1.0,       
    
    eval_strategy="steps",
    eval_steps=training_config['eval_steps'],
    save_steps=training_config['save_steps'],
    save_total_limit=3,
    logging_steps=training_config['logging_steps'],
    seed=42,
    dataloader_num_workers=0,   # Reduced workers
    use_cpu=False,
    report_to="none",
    
    # CRITICAL: Disabled FP16 for numerical stability
    fp16=False,                 # Was True - causing NaN
    bf16=False,
    
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# ===============================================================================
# 5. Training Execution
# ===============================================================================

# print(f"\n🚀 Starting FIXED MIT movie LoRA Fine-tuning...")
# print(f"🎬 Training on {len(syn_train)} movie examples")
# print("-" * 60)

# Clear cache before training
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# Create trainer
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=syn_train,
#     eval_dataset=syn_val,
#     tokenizer=model.data_processor.transformer_tokenizer,
#     data_collator=data_collator,
#     callbacks=[monitor],
# )

# Start training
# train_result = trainer.train()

# ===============================================================================
# 6. FIXED Training Results and Simplified Plots
# ===============================================================================

# print(f"\n🎉 Training Completed!")
# print("=" * 50)

# # Training summary
# print(f"📊 Training Summary:")
# print(f"   • Total steps: {len(monitor.steps)}")
# print(f"   • Best validation loss: {monitor.best_loss:.4f}")
# if monitor.train_losses:
#     print(f"   • Final training loss: {monitor.train_losses[-1]:.4f}")
# if monitor.eval_losses:
#     print(f"   • Final validation loss: {monitor.eval_losses[-1]:.4f}")
# if monitor.gpu_memory:
#     print(f"   • Peak GPU memory: {max(monitor.gpu_memory):.2f} GB")

# # Create simplified plots
# print(f"\n📈 Generating Training Curves...")
# fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# # Plot 1: Training and Validation Loss (with log scale)
# axes[0].plot(monitor.steps, monitor.train_losses, 'b-', alpha=0.7, label='Training Loss')
# axes[0].plot(monitor.eval_steps, monitor.eval_losses, 'r-', marker='o', 
#             linewidth=2, markersize=6, label='Validation Loss')
# axes[0].set_xlabel('Steps')
# axes[0].set_ylabel('Loss')
# axes[0].set_title('Training Progress')
# axes[0].legend()
# axes[0].grid(True, alpha=0.3)
# axes[0].set_yscale('log')  # Log scale for better visualization

# # Plot 2: Learning Rate Schedule
# axes[1].plot(monitor.steps, monitor.learning_rates, 'g-', linewidth=2)
# axes[1].set_xlabel('Steps')
# axes[1].set_ylabel('Learning Rate')
# axes[1].set_title('Learning Rate Schedule')
# axes[1].grid(True, alpha=0.3)
# axes[1].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

# # Plot 3: CPU and GPU Usage on Same Plot (TIME ON X-AXIS)
# if monitor.timestamps:
#     ax3 = axes[2]
    
#     # GPU memory (left y-axis)
#     if monitor.gpu_memory:
#         line1 = ax3.plot(monitor.timestamps, monitor.gpu_memory, 'purple', linewidth=2, label='GPU Memory (GB)')
#         ax3.set_xlabel('Time (minutes)')
#         ax3.set_ylabel('GPU Memory (GB)', color='purple')
#         ax3.tick_params(axis='y', labelcolor='purple')
    
#     # CPU memory (right y-axis)
#     ax3_twin = ax3.twinx()
#     line2 = ax3_twin.plot(monitor.timestamps, monitor.cpu_memory, 'orange', linewidth=2, label='CPU Memory (%)')
#     ax3_twin.set_ylabel('CPU Memory (%)', color='orange')
#     ax3_twin.tick_params(axis='y', labelcolor='orange')
    
#     # Combined legend
#     lines1, labels1 = ax3.get_legend_handles_labels()
#     lines2, labels2 = ax3_twin.get_legend_handles_labels()
#     ax3.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
    
#     ax3.set_title('Resource Usage Over Time')
#     ax3.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.savefig('../models/syn_model/plot.png', dpi=100, bbox_inches='tight')
# plt.show()

# # ===============================================================================
# # 7. Save Model and Test Evaluation
# # ===============================================================================

# # print(f"\n💾 Saving Final Model...")
# # final_model_path = "./models/lora_mit_movie_final"
# # os.makedirs(final_model_path, exist_ok=True)
# # model.save_pretrained(final_model_path)
# # print(f"✅ Model saved to: {final_model_path}")

# model.model.save_pretrained("../models/syn_model")





⚙️ FIXED Training Configuration...
📋 Effective batch size: 8
📋 Total training steps: 1000
🔥 CRITICAL FIXES APPLIED:
   • Learning rate: 0.00021008343694753508 (conservative)
   • LoRA r: 32, alpha: 64 (as requested)
   • FP16: DISABLED for stability


8

In [8]:
# from evaluation.evaluator import enhanced_evaluate
# from evaluation.helper import display_results


# # Test evaluation
# print(f"\n🧪 Final Test Evaluation...")
# model.eval()
# with torch.no_grad():
#     test_results = enhanced_evaluate(
#         model,test_data,entity_types,threshold=0.5,batch_size=8,has_ground_truth=True,logger=logger
#     )

# display_results(test_results)

In [9]:
# test_results["overall_metrics"]["overall_f1_pct"]


In [10]:
from evaluation.evaluator import enhanced_evaluate

no_syn_train_data=[15,25,50,75,100,150,200,350,500,650,750]
f1_scores=[]
con_scores=[]
gliner_f1=[]

for i in no_syn_train_data:

    #data
    total_examples=len(syn_57_examples)
    syn_train=syn_57_examples[:i]
    syn_val=syn_57_examples[750:]
    print(len(syn_train),len(syn_val))


    #model intialize
    model=intialize_model()


    #train
    trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=syn_train,
    eval_dataset=syn_val,
    tokenizer=model.data_processor.transformer_tokenizer,
    data_collator=data_collator,
    #  callbacks=[monitor],
    
    )
    train_result = trainer.train()
    model.model.save_pretrained(f"../models/syn_model_{i}")
    del model, trainer  # DELETE OBJECTS FIRST
    torch.cuda.empty_cache()
    gc.collect()



    
    #eval model load
    model = GLiNER.from_pretrained("knowledgator/modern-gliner-bi-large-v1.0")
    model.config.max_len = 8192  # Change from 2048 to 512
    print(f"Updated max_len to: {model.config.max_len}")
    # FIXED: Set tokenizer max_length to prevent warning                                                                          │ │
    if hasattr(model.data_processor, 'transformer_tokenizer'):    
            model.data_processor.transformer_tokenizer.model_max_length = 8192 
            print(f"Updated tokenizer max_length to: {model.data_processor.transformer_tokenizer.model_max_length}")    
    print("🔧 Loading LoRA adapters...")
    model.model = PeftModel.from_pretrained(model.model, f"../models/syn_model_{i}")
    model.eval()
    model.to('cuda')


    #evals
    with torch.no_grad():
        test_results = enhanced_evaluate(
    model,test_data,entity_types,threshold=0.5,batch_size=8,has_ground_truth=True,logger=logger
    )
        Gliner_results, gliner_f1_score = model.evaluate(
        test_data,
        flat_ner=True,
        threshold=0.5,
        batch_size=16,
        entity_types=entity_types
    )
    f1_score=test_results["overall_metrics"]["overall_f1_pct"]
    con_score=test_results["overall_metrics"]["overall_confidence_pct"]
    
    print(f"f1: {f1_score},Gliner f1:{gliner_f1_score:.2%} ,confidence:{con_score} " )
    f1_scores.append(f1_score)
    con_scores.append(con_score)
    gliner_f1.append(f"{gliner_f1_score:.2%}")
    del model
    torch.cuda.empty_cache()
    gc.collect()


final_no_correct_examples_df=pd.DataFrame({"no_syn_train_data":no_syn_train_data,"f1":f1_scores,"Gliner f1": gliner_f1,"confidence":con_scores})
display(final_no_correct_examples_df)
    

    


15 148


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 51011.81it/s]
INFO:ActiveLearning:Base Parameters: 529,845,248



🔧 Applying FIXED LoRA Configuration...
✅ LoRA applied successfully!
📊 Trainable Parameters: 20,815,872 (3.9% of original)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50283}.


Model after lora


...

There were missing keys in the checkpoint model loaded: ['model.base_model.model.token_rep_layer.bert_layer.model.embeddings.tok_embeddings.weight', 'model.base_model.model.token_rep_layer.bert_layer.model.embeddings.norm.weight', 'model.base_model.model.token_rep_layer.bert_layer.model.layers.0.attn.Wqkv.base_layer.weight', 'model.base_model.model.token_rep_layer.bert_layer.model.layers.0.attn.Wqkv.lora_A.default.weight', 'model.base_model.model.token_rep_layer.bert_layer.model.layers.0.attn.Wqkv.lora_B.default.weight', 'model.base_model.model.token_rep_layer.bert_layer.model.layers.0.attn.Wo.base_layer.weight', 'model.base_model.model.token_rep_layer.bert_layer.model.layers.0.attn.Wo.lora_A.default.weight', 'model.base_model.model.token_rep_layer.bert_layer.model.layers.0.attn.Wo.lora_B.default.weight', 'model.base_model.model.token_rep_layer.bert_layer.model.layers.0.mlp_norm.weight', 'model.base_model.model.token_rep_layer.bert_layer.model.layers.0.mlp.Wi.base_layer.weight', 'model

Updated max_len to: 8192
Updated tokenizer max_length to: 8192
🔧 Loading LoRA adapters...


INFO:ActiveLearning:Running enhanced evaluation...
INFO:ActiveLearning:Processing 2442 examples...
INFO:ActiveLearning:Analyzing errors with ground truth...


f1: 47.92493684590401,Gliner f1:47.92% ,confidence:93.50472255084556 


,no_syn_train_data,f1,Gliner f1,confidence
0,15,53.070658,53.08%,94.696419
1,25,51.666995,51.68%,96.150331
2,50,48.794957,48.80%,97.423565
3,75,47.141019,47.13%,97.018861
4,100,53.578154,53.57%,97.045953
5,150,48.292772,48.29%,97.373458
6,200,49.376243,49.39%,95.528069
7,350,48.135532,48.14%,96.368526
8,500,49.154194,49.18%,95.316450
9,650,48.714503,48.70%,94.663860
